In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import base64  # for logo

from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "password123"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Retrieve ALL records from MongoDB
df = pd.DataFrame.from_records(db.read({}))

# Remove _id column (ObjectId causes DataTable issues)
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load and encode Grazioso Salvare logo
# Make sure the filename matches exactly what is in your Codio code_files directory
image_filename = 'Grazioso Salvare Logo.png'  # adjust if you renamed it
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([
    # Top branding row: logo + unique identifier
    html.Div([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image),
            style={'height': '80px', 'margin-right': '20px'}
        ),
        html.H1('Grazioso Salvare Dashboard - Aaron Jones')
    ], style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'}),
    
    html.Hr(),

    # ---------- Filter Controls ----------
    html.Div([
        html.H3("Rescue Type Filter"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset - Show All Animals', 'value': 'reset'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
            ],
            value='reset',
            labelStyle={'display': 'block'}
        )
    ], style={'width': '25%', 'margin': '0 auto'}),
    
    html.Hr(),

    # ---------- Data Table ----------
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        row_selectable="single",
        selected_rows=[0],  # default first row
        style_table={'overflowX': 'auto', 'maxHeight': '400px', 'overflowY': 'scroll'},
        style_cell={
            'minWidth': '80px', 'width': '150px', 'maxWidth': '200px',
            'whiteSpace': 'normal'
        },
    ),

    html.Br(),
    html.Hr(),

    # ---------- Row for Chart + Map ----------
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

# Rescue-type filter → DataTable data
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """
    Update the table based on the rescue-type filter.
    Uses MongoDB queries via the CRUD module.
    """
    # Base: default to all records
    if filter_type == 'reset' or filter_type is None:
        dff = df.copy()
    else:
        # Build MongoDB query using CS-340 rescue profiles
        query = {"animal_type": "Dog"}
        
        if filter_type == 'water':
            # Example profile for water rescue
            query.update({
                "breed": {"$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever Mix",
                    "Newfoundland Mix"
                ]},
                "sex_upon_outcome": "Intact Female",
                "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
            })
        elif filter_type == 'mountain':
            # Wilderness rescue profile (matches your earlier quiz)
            query.update({
                "breed": {"$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]},
                "sex_upon_outcome": "Intact Male",
                "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
            })
        elif filter_type == 'disaster':
            # Disaster / individual tracking profile
            query.update({
                "breed": {"$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]},
                "sex_upon_outcome": "Intact Male",
                "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
            })
        else:
            dff = df.copy()
            return dff.to_dict('records')

        # Actually query MongoDB through the CRUD module
        results = db.read(query)
        dff = pd.DataFrame.from_records(results)
        if '_id' in dff.columns:
            dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# DataTable → Pie chart
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    """
    Build a pie chart of breed counts for whatever is currently in the table
    (filtered or unfiltered).
    """
    if viewData is None or len(viewData) == 0:
        return [html.Div("No data to display")]

    dff = pd.DataFrame.from_dict(viewData)

    # Count by breed
    if 'breed' not in dff.columns:
        return [html.Div("Breed field not found in data")]

    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']

    fig = px.pie(
        breed_counts,
        names='breed',
        values='count',
        title='Breed Distribution for Current Filter'
    )

    return [dcc.Graph(figure=fig)]


# Highlight selected column in table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]


# DataTable selection → Map
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    """
    Show a marker for the selected animal (or first row) on the map.
    """
    if viewData is None or len(viewData) == 0:
        return [html.Div("No data available for map")]

    dff = pd.DataFrame.from_dict(viewData)

    # Default to first row if nothing selected
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Safety check on bounds
    if row >= len(dff):
        row = 0

    # Use column names; dataset has location_lat & location_long
    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']
    breed = dff.iloc[row]['breed']
    name = dff.iloc[row]['name']

    # Austin TX approximate center
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(name)
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode
app.run_server()


Dash app running on https://cargonavy-nepalapropos-3000.codio.io/proxy/8050/
